In [1]:
# WorkFlow
'''
- ㅇ 데이터 준비
    - UCI Opinion Review 데이터셋 로드 (텍스트 리뷰 + 레이블)
    - 텍스트 전처리: 소문자화, 불용어 제거, 토큰화, Lemmatization
- ㅇ 벡터화 (Feature Engineering)
    - TF-IDF 벡터화 (scikit-learn TfidfVectorizer)
    - 또는 Word2Vec / BERT 임베딩 활용 (더 높은 성능 가능)
- ㅇ PyCaret 군집화
    - from pycaret.clustering import *
    - setup(data, normalize=True, ignore_features=[‘text’])
    - compare_models() → 가장 성능 좋은 군집화 모델 자동 선택 (예: KMeans, DBSCAN, HDBSCAN 등)
- ㅇ 문서 유사도 분석
    - 군집화된 결과를 바탕으로 같은 클러스터 내 문서들 간 유사도 계산
    - cosine_similarity(tfidf_matrix) 활용
    - 클러스터별 대표 문서(centroid)와 다른 문서 간 유사도 측정
- ㅇ 결과 해석
    - 어떤 클러스터가 긍정/부정 리뷰를 잘 구분하는지 확인
    - 클러스터 내 문서 유사도 상위 문서들을 추출하여 대표 리뷰 선정
'''

'\n- ㅇ 데이터 준비\n    - UCI Opinion Review 데이터셋 로드 (텍스트 리뷰 + 레이블)\n    - 텍스트 전처리: 소문자화, 불용어 제거, 토큰화, Lemmatization\n- ㅇ 벡터화 (Feature Engineering)\n    - TF-IDF 벡터화 (scikit-learn TfidfVectorizer)\n    - 또는 Word2Vec / BERT 임베딩 활용 (더 높은 성능 가능)\n- ㅇ PyCaret 군집화\n    - from pycaret.clustering import *\n    - setup(data, normalize=True, ignore_features=[‘text’])\n    - compare_models() → 가장 성능 좋은 군집화 모델 자동 선택 (예: KMeans, DBSCAN, HDBSCAN 등)\n- ㅇ 문서 유사도 분석\n    - 군집화된 결과를 바탕으로 같은 클러스터 내 문서들 간 유사도 계산\n    - cosine_similarity(tfidf_matrix) 활용\n    - 클러스터별 대표 문서(centroid)와 다른 문서 간 유사도 측정\n- ㅇ 결과 해석\n    - 어떤 클러스터가 긍정/부정 리뷰를 잘 구분하는지 확인\n    - 클러스터 내 문서 유사도 상위 문서들을 추출하여 대표 리뷰 선정\n'

In [2]:
import os
import sys
import glob

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from pycaret.clustering import setup, create_model, assign_model, models, pull
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from datetime import datetime
import warnings
import time

from utils import preprocessing

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\tj\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
warnings.filterwarnings('ignore')
DATA_PATH = '../data'
OUTPUT_DIR = '../results'
IMAGE_DIR = '../images'
    
document_df = preprocessing.get_default_data()


📂 51개 파일 발견
🔄 텍스트 전처리 시작...
   옵션: HTML제거=True, URL제거=True, 숫자제거=True
   옵션: 불용어제거=True, Lemmatization=True, Stemming=False
✅ 전처리 완료:
   - 원본 문서 수: 51
   - 제거된 빈 문서: 0
   - 최종 문서 수: 51
   - 평균 단어 수: 1266.9


In [5]:
print(document_df.columns)

Index(['filename', 'opinion_text', 'processed_text', 'word_count'], dtype='object')


In [6]:
# 전처리된 텍스트 사용
texts = document_df['processed_text']

# TF-IDF 벡터화
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(texts)

# PyCaret 입력용 DataFrame 변환
tfidf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

In [11]:
from pycaret.clustering import setup, create_model, assign_model, plot_model

# 세팅
clu = setup(tfidf_df, session_id=123, normalize=True)

,Description,Value
0,Session id,123
1,Original data shape,"(51, 5000)"
2,Transformed data shape,"(51, 5000)"
3,Numeric features,5000
4,Preprocess,True
5,Imputation type,simple
6,Numeric imputation,mean
7,Categorical imputation,mode
8,Normalize,True
9,Normalize method,zscore


In [12]:
# 모델 목록 확인
from pycaret.clustering import models
available_models = models()
print(available_models)

                                       Name  \
ID                                            
kmeans                   K-Means Clustering   
ap                     Affinity Propagation   
meanshift             Mean Shift Clustering   
sc                      Spectral Clustering   
hclust             Agglomerative Clustering   
dbscan     Density-Based Spatial Clustering   
optics                    OPTICS Clustering   
birch                      Birch Clustering   
kmodes                   K-Modes Clustering   

                                                   Reference  
ID                                                            
kmeans                        sklearn.cluster._kmeans.KMeans  
ap         sklearn.cluster._affinity_propagation.Affinity...  
meanshift              sklearn.cluster._mean_shift.MeanShift  
sc              sklearn.cluster._spectral.SpectralClustering  
hclust     sklearn.cluster._agglomerative.AgglomerativeCl...  
dbscan                        sklearn.clu

In [ ]:
# 모델 생성 및 할당
from pycaret.clustering import create_model, assign_model


kmeans = create_model('kmeans')
clustered_df = assign_model(kmeans)

,Silhouette,Calinski-Harabasz,Davies-Bouldin,Homogeneity,Rand Index,Completeness
0,-0.0369,1.1096,0.9635,0,0,0
